# Mapping Change Around Qatar's World Cup Stadiums

An automated ArcPy pipeline that measures visible change around all eight of Qatar's 2022 World Cup stadiums, using Sentinel-2 imagery from three time windows: August-October 2016 (pre-construction boom), 2022 (tournament year), and 2025 (three years post-tournament). Rather than building a separate map layout by hand for each stadium, the pipeline scripts a single ArcGIS Pro Spatial Map Series that dynamically re-labels and re-frames itself per stadium, exporting the full set as one reproducible PDF.

All paths are built with `os.path.join()` relative to the ArcGIS Pro project's home folder, so the notebook runs unmodified on any machine with the same project structure.

## Setup

In [50]:
import arcpy
import os
from arcpy.sa import RasterCalculator, Grayscale

## Loading the imagery

Three Sentinel-2 RGB composites are loaded into the active map: one from each of the three time windows being compared.

In [34]:
#Now, we need to identify our current arcgis project
aprx = arcpy.mp.ArcGISProject("CURRENT") #aprx == arcgis project file
map_obj = aprx.activeMap #the current active map
project_folder = aprx.homeFolder #set home folder

# define our dataset names
#we are using three .tif files, specify extension 
datasets = ["S2_RGB_2016-08-01_to_2016-10-31.tif", 
            "S2_RGB_2022-08-01_to_2022-10-31.tif",
            "S2_RGB_2025-08-01_to_2025-10-13.tif"]

for data_file in datasets:
    # define data path for reproducibility
    data_path = os.path.join(project_folder, data_file)
    
    # add data from the path 
    map_obj.addDataFromPath(data_path)
    
    print(f"Successfully added: {data_file}")

Successfully added: S2_RGB_2016-08-01_to_2016-10-31.tif
Successfully added: S2_RGB_2022-08-01_to_2022-10-31.tif
Successfully added: S2_RGB_2025-08-01_to_2025-10-13.tif


## Change detection: 2016 to 2022

The absolute pixel difference between the 2016 and 2022 composites highlights where construction activity was concentrated in the run-up to the tournament.

In [130]:
#restating these just to be sure
arcpy.env.overwriteOutput = True
aprx = arcpy.mp.ArcGISProject("CURRENT")
project_folder = aprx.homeFolder 

#here we are using the raster calculator function to find the difference between 2016 and 2022
#os.path.join for reproducibility
raster_2022 = os.path.join(project_folder, "S2_RGB_2022-08-01_to_2022-10-31.tif")
raster_2016 = os.path.join(project_folder, "S2_RGB_2016-08-01_to_2016-10-31.tif")

#specify output and output paths
raster_output = "Absolute_difference_2022_2016.tif"
final_output_path = os.path.join(project_folder, raster_output)

#do absolute difference using raster calculator 
abs_difference = RasterCalculator ([raster_2022, raster_2016], ["x", "y"], "abs(x - y)")

#added this if statement due to updating issues
if arcpy.Exists(final_output_path): 
    arcpy.management.Delete(final_output_path)

abs_difference.save(final_output_path)

In [5]:
#here we are loading in our new raster 
output_raster_name = "Absolute_difference_2022_2016.tif"
saved_raster_path_2022 = os.path.join(project_folder, output_raster_name)

map_obj.addDataFromPath(saved_raster_path_2022)

Converting the difference raster to grayscale gives a single-band change-intensity map. `arcpy.sa.Grayscale` was used here in place of manually summing the RGB bands, after running into syntax limitations trying to do the summation directly in ArcPy.

In [131]:
#here we are converting to grayscale 
arcpy.env.overwriteOutput = True

#input raster
input_raster = arcpy.Raster (saved_raster_path_2022)

#output raster (specify .tif)
grayscale_raster_name = "2016-2022.tif"
grayscale_output_path_2022 = os.path.join(project_folder, grayscale_raster_name)

grayscale_raster = Grayscale (input_raster)

if arcpy.Exists(grayscale_output_path_2022):
    arcpy.management.Delete(grayscale_output_path_2022)

#save and add data
grayscale_raster.save(grayscale_output_path_2022)
map_obj.addDataFromPath(grayscale_output_path_2022)

## Change detection: 2022 to 2025

The same process, run on the 2022 and 2025 composites, to see what changed in the years immediately following the tournament.

In [9]:
arcpy.env.overwriteOutput = True
#here we are using the raster calculator function to find the difference between 2025 and 2022
raster_output_2025 = "Absolute_difference_2025_2022.tif"

# Combine the folder path and filename for the final path
output_path = os.path.join(project_folder, raster_output_2025)

#define raster names
raster_2022 = "S2_RGB_2022-08-01_to_2022-10-31.tif"
raster_2025 = "S2_RGB_2025-08-01_to_2025-10-13.tif"

#raster calculator
abs_difference_2025 = RasterCalculator ([raster_2025, raster_2022], ["x", "y"], "abs(x - y)")

abs_difference_2025.save(output_path)

#here we are loading in our new raster 
output_raster_name = "Absolute_difference_2025_2022.tif"
saved_raster_path_2025 = os.path.join(project_folder, raster_output_2025)

map_obj.addDataFromPath(saved_raster_path_2025)

In [10]:
arcpy.env.overwriteOutput = True
##convert 2025 to grayscale 
#same process as before
input_raster_2025 = arcpy.Raster (saved_raster_path_2025)

grayscale_raster_name_2025 = "2022-2025.tif"
grayscale_output_path_2025 = os.path.join(project_folder, grayscale_raster_name_2025)

grayscale_raster_2025 = Grayscale(input_raster_2025)

if arcpy.Exists(grayscale_output_path_2025):
    arcpy.Delete_management(grayscale_output_path_2025)

grayscale_raster_2025.save(grayscale_output_path_2025)
map_obj.addDataFromPath(grayscale_output_path_2025)

*An earlier version of this notebook built each stadium's map as a separate, individually laid-out page. That approach worked but didn't scale - it's replaced below by a single scripted map series that handles all seven stadiums at once.*

## Sourcing and preparing the stadium locations

Official stadium locations are pulled directly from an ArcGIS Online feature service rather than digitized by hand, which keeps the dataset authoritative and easy to refresh.

In [24]:
#load in data from arcgis online
#use join and copy features so we can edit it 
#add "/0" to the end of URL so it will load properly 
stadium_path = "https://services.arcgis.com/8df8p0NlLFEShl0r/arcgis/rest/services/Qatar_Stadiums_WorldCup2022/FeatureServer/0"
output_path_stadiums = os.path.join(project_folder, "Qatar_Stadiums")
arcpy.management.CopyFeatures(stadium_path, output_path_stadiums)

<Result 'Qatar_Stadiums.shp'>

Al Bayt Stadium sits well north of the other seven and would distort the shared map extent if included in the same series, so it's excluded here and would need its own map if added later.

In [39]:
##clean stadium file, specify os.path.join for reproducibility
input_stadium_file = os.path.join(project_folder, "Qatar_Stadiums")
output_stadium_file = os.path.join(project_folder, "Qatar_Stadiums_Cleaned")

#select all features except northernmost stadium
where_clause = "Stadium_Na <> 'Al Bayt Stadium'"

#use select feature, we already have out output file defined 
arcpy.analysis.Select(input_stadium_file, output_stadium_file, where_clause)

<Result 'Qatar_Stadiums_Cleaned.shp'>

## Buffering the stadiums for consistent map framing

Zooming the map series directly to each stadium point proved unreliable, so each point is buffered by 1000 meters and that buffer polygon is used as the index layer instead - giving the series a consistent, predictable extent to frame around at each stadium.

In [103]:
##was having issues defining the path, used {}.shp and it worked
stadium_feature_class_path = f"{output_stadium_file}.shp"

## Building and exporting the automated map series

This is the core of the pipeline: a single ArcGIS Pro layout, driven by a Spatial Map Series indexed on the buffer layer above, that iterates through all seven stadiums automatically. For each page, the title text dynamically updates to the current stadium's name and the map frame re-centers on it.

A couple of implementation notes worth flagging:
- Editing the stadium points' symbology directly kept breaking the series' indexing, so a duplicate copy of the layer is used purely for the visual highlight marker, leaving the layer the series actually indexes on untouched.
- The layout construction (title placement, north arrow, scale bar, legend) is adapted from ESRI's official layout-automation documentation and a course demonstration, then modified for this series.
- AI assistance was used to work out the correct symbology syntax for the highlight marker, where documentation was sparse.

The final export produces all seven stadium pages as a single 300 dpi PDF.

In [129]:
#define field of interest 
#this took awhile because I discovered that when i ran my code on a different computer, field names were also changed (?)
field_of_interest = 'Stadium_Na' 
#create the path for pdf output 
output_pdf_path = os.path.join(project_folder, 'Stadium_Map_Series.pdf')

#create map and define layer 
m = p.createMap('New Map', 'MAP')
#these are our stadium points
lyr = m.addDataFromPath(stadium_feature_class_path)
#this is our background change map 
change_lyr = m.addDataFromPath(grayscale_output_path_2022)
#this is our buffer layer that we just created
buffer_lyr = m.addDataFromPath(layer_file)
buffer_lyr.name = "Stadium Buffer"


#downstream effects with labeling, indexing, etc. In hindsight this may have not been the best method but I'm afraid to change it now
highlight_lyr_copy = m.insertLayer(lyr, lyr, 'AFTER')
highlight_lyr_copy.name = "Stadium Highlight"

#this is editing symbology for the highlight layer to make larger more visible points
sym = highlight_lyr_copy.symbology
if hasattr (sym, 'renderer') and sym.renderer.type == 'SimpleRenderer':
    sym.renderer.symbol.applySymbolFromGallery('Circle 1', 0)
    sym.renderer.symbol.size = 15.0
    sym.renderer.symbol.color = {'RGB':[250, 0, 0, 80]}
    highlight_lyr_copy.symbology = sym

lyt = p.createLayout(8.5, 11, 'INCH') #create a new 8.5" x 11" layout
mf = lyt.createMapFrame(MakeRec_LL(0.5,2.0,7.5,8), m, "New Map Frame") 

sms = lyt.createSpatialMapSeries(mapframe = mf, index_layer = buffer_lyr, name_field = "Stadium_Na")
sms.scale = 3000
sms.clipToIndexFeature = False
lyr.visible = False
buffer_lyr.visible = False
lyr.minScale = 0
lyr.maxScale = 0


highlight_lyr_copy.pageDefinitionEnabled = True
highlight_lyr_copy.pageDefinitionType = 'MATCH'
highlight_lyr_copy.pageDefinitionField = field_of_interest


#Create point text element using a system style item
txtStyleItem = p.listStyleItems('ArcGIS 2D', 'TEXT', 'Title (Serif)')[0]
ptTxt = p.createTextElement(lyt, arcpy.Point(4.25, 10.5), 'POINT',
                            'Grayscale Absolute Change Map Qatar 2016-2022',
                            10, style_item=txtStyleItem)

ptTxt.setAnchor('Center_Point')
ptTxt.elementPositionX = 4.25
ptTxt.elementPositionY = 10.5

while ptTxt.elementWidth < (lyt.pageWidth - (lyt.pageWidth * 0.1)):
  ptTxt.textSize = ptTxt.textSize + 0.1

naStyle = p.listStyleItems('ArcGIS 2D', 'North_Arrow', 'Compass North 1')[0]
#place it on the map
na = lyt.createMapSurroundElement(arcpy.Point(7,1.0), 'North_Arrow', mf, naStyle, "Compass North Arrow")
na.elementWidth = 0.5

sbName = 'Scale Line 1' #name of scale bar style in ArcGIS
sbStyle = p.listStyleItems('ArcGIS 2D', 'Scale_bar', sbName)[0] #similar to north arrow search, finds a scale bar with above name & selects it
sbEnv = MakeRec_LL(0.5, 1.65, 2.5, 0.5) #create a rectangle for the scale bar
sb = lyt.createMapSurroundElement(sbEnv, 'Scale_bar', mf, sbStyle, 'New Scale Bar') #place the scale bar

lgName = "legend!" #create a legend!
lgStyle = p.listStyleItems('ArcGIS 2D', 'LEGEND', 'legend 3')[0] #search for the legend 3 stlye
lgEnv = arcpy.Extent(2.8, 0.4, 6, 1.4) #created rectangle for legend
lg = lyt.createMapSurroundElement(lgEnv, 'LEGEND', mf, lgStyle, 'New legend') #create the legend on the layout

txtStyleItem = p.listStyleItems('ArcGIS 2D', 'TEXT', 'Title (Serif)')[0]
stadium_text_tag = f"<dyn type='page' property='name'/>"

stadiumTxt = p.createTextElement(lyt, arcpy.Point(4.25, 10.1), 'POINT',
                                 stadium_text_tag,
                                 12, style_item=txtStyleItem)
stadiumTxt.setAnchor('Center_Point')
stadiumTxt.elementPositionX = 4.25
stadiumTxt.elementPositionY = 10.1

lyt.openView()

output_dir = os.path.dirname(output_pdf_path)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

sms.exportToPDF(output_pdf_path, "ALL", resolution=300)

Dynamic Stadium Label added.
Exporting map series to: Output\Stadium_Map_Series.pdf
Export complete.
